# 🎭 Multi-Speaker Drama & Podcast MVP — Qwen3-TTS
**Purpose:** Turn a script with character tags into a full multi-character audio production.

## Script Format Guide
Use the `[Speaker Name]: dialogue line` syntax. Example:
- `[Host]: Welcome back everyone!` 🎙️
- `[Guest]: Thanks for having me.` 👋

In [ ]:
!pip install -q qwen-tts soundfile

import torch
import soundfile as sf
import os
import gc
import numpy as np
import re
from IPython.display import Audio, display


In [ ]:
def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()

MODEL_SIZE = "1.7B"
DTYPE = torch.bfloat16
OUTPUT_DIR = "/content/podcast_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
from qwen_tts import Qwen3TTSModel

print("Loading CustomVoice model...")
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"
cv_model = Qwen3TTSModel.from_pretrained(
    model_id,
    device_map="cuda:0",
    dtype=DTYPE,
    attn_implementation="sdpa"
)

print("\nAvailable Preset Speakers:")
print(cv_model.get_supported_speakers())


In [ ]:
PODCAST_SCRIPT = """
[Host]: Welcome back to AI Futures! Today we have two amazing guests joining us.
[Tech Guest]: Thanks for having me, really excited to be here and talk tech.
[Science Guest]: Same here. The recent breakthroughs in fundamental science are just mind-blowing.
[Host]: Absolutely. Let's start with you. What do you think is the biggest shift this year?
[Tech Guest]: Oh, without a doubt, it's the convergence of multimodal models and agentic workflows.
[Science Guest]: I agree, but from a physics perspective, AI accelerating materials discovery is huge.
[Host]: We've seen AI find new crystal structures. How does that impact everyday people?
[Science Guest]: It means better batteries, stronger materials, and completely new tech we haven't even dreamt of yet.
[Tech Guest]: Exactly. And when those materials meet robotics, the sky is the limit.
[Host]: Amazing insights. Thanks for tuning in, folks!
"""

SPEAKER_MAP = {
    "Host": {"speaker": "Ryan", "language": "English", "instruct": "Enthusiastic and welcoming podcast host"},
    "Tech Guest": {"speaker": "Stella", "language": "English", "instruct": "Clear, professional, and slightly fast-paced tech expert"},
    "Science Guest": {"speaker": "David", "language": "English", "instruct": "Calm, thoughtful, and articulate scientist"}
}

def parse_script(script):
    lines = script.strip().split('\n')
    parsed = []
    for line in lines:
        match = re.match(r'\[(.*?)\]:\s*(.*)', line)
        if match:
            speaker, text = match.groups()
            parsed.append((speaker.strip(), text.strip()))
    return parsed

parsed_lines = parse_script(PODCAST_SCRIPT)
print(f"Parsed {len(parsed_lines)} lines of dialogue.")


In [ ]:
print("Generating multi-speaker audio...")
all_audio = []
sr = 24000
speaker_stats = {name: {"lines": 0, "samples": 0} for name in SPEAKER_MAP.keys()}

for i, (speaker, text) in enumerate(parsed_lines):
    print(f"Line {i+1} [{speaker}]: Generating...")
    config = SPEAKER_MAP.get(speaker, SPEAKER_MAP["Host"])
    
    audio, _ = cv_model.generate_custom_voice(
        text=text,
        language=config["language"],
        speaker=config["speaker"],
        instruct=config["instruct"]
    )
    
    # Save individual line
    line_filename = f"{i+1:02d}_{speaker.replace(' ', '_')}.wav"
    sf.write(os.path.join(OUTPUT_DIR, line_filename), audio, sr)
    
    all_audio.append(audio)
    if speaker in speaker_stats:
        speaker_stats[speaker]["lines"] += 1
        speaker_stats[speaker]["samples"] += len(audio)
        
    # 200ms silence between turns
    pause = np.zeros(int(sr * 0.2), dtype=np.float32)
    all_audio.append(pause)

combined_audio = np.concatenate(all_audio)
final_path = os.path.join(OUTPUT_DIR, "podcast_episode.wav")
sf.write(final_path, combined_audio, sr)
print(f"\nSaved full podcast to {final_path}")


In [ ]:
print("--- Podcast Episode ---")
display(Audio(final_path))

print("\n--- Speaker Breakdown ---")
print(f"{'Speaker':<15} | {'Lines':<5} | {'Duration (s)':<10}")
print("-" * 40)
for spk, stats in speaker_stats.items():
    duration = stats["samples"] / sr
    print(f"{spk:<15} | {stats['lines']:<5} | {duration:.2f}")


In [ ]:
import shutil

try:
    from google.colab import files
    print("Zipping outputs...")
    shutil.make_archive("/content/podcast_outputs", 'zip', OUTPUT_DIR)
    print("Downloading zip file...")
    files.download("/content/podcast_outputs.zip")
except ImportError:
    print("google.colab import failed. Note: The download cell only works in Google Colab.")
